# Level 3 Anomaly Modeling Experiments

This notebook evaluates `is_forged` predictions and inspects confidence outputs on the public dummy set.

In [ ]:
from pathlib import Path
import json
from solution import DocFusionSolution

repo = Path('/Users/abx/rihal/rihal-codestacker/ML')
submission = repo / 'my_submission'
sol = DocFusionSolution()
model_dir = sol.train(str(repo / 'dummy_data' / 'train'), str(submission / 'models' / 'nb_level3'))
pred_path = submission / 'models' / 'nb_level3' / 'predictions.jsonl'
sol.predict(model_dir, str(repo / 'dummy_data' / 'test'), str(pred_path))
model_dir, pred_path

In [ ]:
preds = {}
for line in pred_path.read_text().splitlines():
    r = json.loads(line)
    preds[r['id']] = r

labels = {}
for line in (repo / 'dummy_data' / 'test' / 'labels.jsonl').read_text().splitlines():
    r = json.loads(line)
    labels[r['id']] = int(r['label']['is_forged'])

rows = []
for rid, y in sorted(labels.items()):
    p = int(preds[rid]['is_forged'])
    rows.append((rid, y, p, int(y == p)))

rows

In [ ]:
acc = sum(x[3] for x in rows) / len(rows)
precision = sum(int(y == 1 and p == 1) for _, y, p, _ in rows) / max(1, sum(int(p == 1) for _, _, p, _ in rows))
recall = sum(int(y == 1 and p == 1) for _, y, p, _ in rows) / max(1, sum(int(y == 1) for _, y, _, _ in rows))

{'accuracy': acc, 'precision': precision, 'recall': recall}